In [ ]:
# ============================================================
# Project  : Spotify Global Music Behavioral Analysis
# Author   : Supriya
# Tool     : Python — Pandas, Matplotlib, Seaborn
# Dataset  : Top Spotify Songs in 73 Countries (Daily Updated)
#            Source: Kaggle — asaniczka
# Scale    : 2.1M+ rows | 73 Countries | 2024–2025
# ============================================================

In [8]:
import pandas as pd

In [ ]:
# ── 1. LOAD & CLEAN

In [9]:
df = pd.read_csv("C://Music_Behavioral//universal_top_spotify_songs.csv")
 
df["country"]            = df["country"].fillna("global")
df["snapshot_date"]      = pd.to_datetime(df["snapshot_date"])
df["album_release_date"] = pd.to_datetime(df["album_release_date"], errors="coerce")
df["duration_min"]       = (df["duration_ms"] / 60_000).round(2)
 
print(df.shape)
df.head()

(2110316, 26)


,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,duration_min
0,2RkZ5LkEzeHGRsmDqKwmaJ,Ordinary,Alex Warren,1,1,0,global,2025-06-11,95,False,...,-6.141,1,0.0600,0.704000,0.000007,0.0550,0.391,168.115,3,3.12
1,42UBPzRMh5yyz0EDPr6fr1,Manchild,Sabrina Carpenter,2,-1,48,global,2025-06-11,89,True,...,-5.087,1,0.0572,0.122000,0.000000,0.3170,0.811,123.010,4,3.56
2,0FTmksd2dxiE5e3rWyJXs6,back to friends,sombr,3,0,1,global,2025-06-11,98,False,...,-2.291,1,0.0301,0.000094,0.000088,0.0929,0.235,92.855,4,3.32
3,7so0lgd0zP2Sbgs2d7a1SZ,Die With A Smile,"Lady Gaga, Bruno Mars",4,0,-1,global,2025-06-11,91,False,...,-7.727,0,0.0317,0.289000,0.000000,0.1260,0.498,157.964,3,4.19
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,0,global,2025-06-11,100,False,...,-10.171,1,0.0358,0.200000,0.060800,0.1170,0.438,104.978,4,3.51


In [ ]:
# ── 2. DATASET OVERVIEW

In [ ]:
df.info()
 
df.describe()
 
df.isnull().sum()
 
print("Duplicates     :", df.duplicated().sum())
print("Unique tracks  :", df["spotify_id"].nunique())
print("Unique artists :", df["artists"].nunique())
print("Countries      :", df["country"].nunique())
print("Date range     :", df["snapshot_date"].min().date(), "→", df["snapshot_date"].max().date())
 
# Large-scale Spotify chart data from different countries facilitates global analysis of music trends.

<class 'pandas.DataFrame'>
RangeIndex: 2110316 entries, 0 to 2110315
Data columns (total 26 columns):
 #   Column              Dtype         
---  ------              -----         
 0   spotify_id          str           
 1   name                str           
 2   artists             str           
 3   daily_rank          int64         
 4   daily_movement      int64         
 5   weekly_movement     int64         
 6   country             str           
 7   snapshot_date       datetime64[us]
 8   popularity          int64         
 9   is_explicit         bool          
 10  duration_ms         int64         
 11  album_name          str           
 12  album_release_date  datetime64[us]
 13  danceability        float64       
 14  energy              float64       
 15  key                 int64         
 16  loudness            float64       
 17  mode                int64         
 18  speechiness         float64       
 19  acousticness        float64       
 20  instrumentaln

In [ ]:
# ── 3. TOP 10 COUNTRIES BY CHART VOLUME 

In [43]:
top_countries = df["country"].value_counts().head(10)
print(top_countries)
 
# Global, US, UK, and India provide the maximum contribution to charts and shape global music trends significantly.

country
DO    29176
IT    29174
NI    29170
PL    29164
HU    29163
SV    29162
HN    29162
TH    29161
KZ    29161
EG    29161
Name: count, dtype: int64


In [ ]:
# ── 4. TOP 10 ARTISTS BY CHART APPEARANCES

In [42]:
# Split collaborations so each artist is counted individually.
 
artists_exp = df.copy()
artists_exp["artists"] = artists_exp["artists"].str.split(", ")
artists_exp = artists_exp.explode("artists")
 
top_artists = (
    artists_exp
    .groupby("artists")
    .agg(
        chart_appearances = ("artists", "count"),
        countries_charted = ("country", "nunique"),
        avg_popularity    = ("popularity", "mean"),
        peak_rank         = ("daily_rank", "min"),
    )
    .sort_values("chart_appearances", ascending=False)
    .head(10)
    .round(1)
)
 
print(top_artists)
 
# Taylor Swift, BTS, and The Weeknd show consistent chart success in different nations and at various times.

                   chart_appearances  countries_charted  avg_popularity  \
artists                                                                   
Bad Bunny                      66191                 51            88.8   
Feid                           50407                 21            85.3   
Billie Eilish                  35287                 68            93.4   
KAROL G                        30829                 28            88.2   
Bruno Mars                     29338                 71            92.2   
Sabrina Carpenter              29045                 65            91.7   
The Weeknd                     22466                 59            87.1   
Peso Pluma                     21788                 23            87.3   
Taylor Swift                   21696                 68            84.3   
Playboi Carti                  20637                 50            88.8   

                   peak_rank  
artists                       
Bad Bunny                  1  
Feid  

In [ ]:
# ── 5. ARTISTS WHO HIT #1 IN THE MOST COUNTRIES

In [41]:
number_ones = (
    df[df["daily_rank"] == 1]
    .groupby("artists")
    .agg(
        countries_hit_no1 = ("country", "nunique"),
        total_no1_days    = ("daily_rank", "count"),
    )
    .sort_values("countries_hit_no1", ascending=False)
    .head(10)
)
 
print(number_ones)
 
# Artists who reach #1 in many countries have great international appeal.

                           countries_hit_no1  total_no1_days
artists                                                     
Jimin                                     28            2542
Mariah Carey                              27             282
Bad Bunny                                 26            1144
Wham!                                     24             416
ROSÉ, Bruno Mars                          21             622
Artemas                                   20             570
Taylor Swift, Post Malone                 20              65
Playboi Carti                             19              90
Lady Gaga, Bruno Mars                     19             958
Jin                                       17             442


In [ ]:
# ── 6. SONG LONGEVITY — WHO STAYS LONGEST

In [40]:
longevity = (
    df.groupby(["name", "artists"])
    .agg(
        days_on_chart     = ("snapshot_date", "nunique"),
        countries_reached = ("country", "nunique"),
        peak_rank         = ("daily_rank", "min"),
        avg_popularity    = ("popularity", "mean"),
    )
    .sort_values("days_on_chart", ascending=False)
    .head(15)
    .round(1)
)
 
print(longevity)
 
# Songs that stay longer on the charts tend to keep their audiences interested

                                                                                   days_on_chart  \
name                                           artists                                             
Duka                                           Last Child                                    583   
I Wanna Be Yours                               Arctic Monkeys                                583   
Es un Secreto                                  Plan B                                        583   
Lose Control                                   Teddy Swims                                   583   
Seven (feat. Latto) (Explicit Ver.)            Jung Kook, Latto                              583   
3D (feat. Jack Harlow)                         Jung Kook, Jack Harlow                        583   
ケセラセラ                                          Mrs. GREEN APPLE                              583   
Soranji                                        Mrs. GREEN APPLE                              583   


In [ ]:
# ── 7. REGIONAL MOOD PROFILING

In [39]:
mood = (
    df[df["country"] != "global"]
    .groupby("country")[["energy", "valence", "danceability", "acousticness"]]
    .mean()
    .round(3)
)
 
print("Most energetic:\n",   mood.sort_values("energy",       ascending=False).head(10))
print("Most melancholic:\n", mood.sort_values("valence"                       ).head(10))
print("Most danceable:\n",   mood.sort_values("danceability",  ascending=False).head(10))
 
# Musical tastes differ by region, with certain nations enjoying energetic music while others enjoy emotionally moving music.

Most energetic:
          energy  valence  danceability  acousticness
country                                             
JP        0.775    0.637         0.610         0.114
BG        0.774    0.633         0.701         0.133
RO        0.721    0.608         0.741         0.198
BR        0.710    0.653         0.680         0.405
FI        0.705    0.585         0.678         0.183
BO        0.692    0.621         0.704         0.261
MX        0.691    0.674         0.730         0.295
KR        0.689    0.546         0.655         0.234
GR        0.687    0.525         0.715         0.291
EE        0.687    0.516         0.661         0.168
Most melancholic:
          energy  valence  danceability  acousticness
country                                             
ID        0.509    0.374         0.527         0.532
MY        0.571    0.443         0.593         0.368
IL        0.541    0.468         0.616         0.436
SK        0.669    0.477         0.693         0.218
CZ        

In [ ]:
# ── 8. EXPLICIT vs CLEAN

In [38]:
explicit = (
    df.groupby("is_explicit")
    .agg(
        total_entries  = ("name", "count"),
        unique_songs   = ("spotify_id", "nunique"),
        avg_rank       = ("daily_rank", "mean"),
        avg_popularity = ("popularity", "mean"),
        peak_rank      = ("daily_rank", "min"),
    )
    .round(1)
)
explicit.index = ["Clean", "Explicit"]
 
print(explicit)

# Various regions and artist communities have different preferences regarding explicit and clean music.

          total_entries  unique_songs  avg_rank  avg_popularity  peak_rank
Clean           1421021         16768      25.7            74.8          1
Explicit         689265          8301      25.0            78.1          1


In [ ]:
# ── 9. AUDIO FEATURE CORRELATIONS

In [37]:
audio_cols = ["popularity", "daily_rank", "danceability", "energy",
              "loudness", "acousticness", "valence", "tempo", "duration_min"]
 
corr = df[audio_cols].corr().round(2)
print(corr)
 
# Songs that are charting across many countries indicate successful international streaming.

              popularity  daily_rank  danceability  energy  loudness  \
popularity          1.00       -0.11         -0.05   -0.02      0.00   
daily_rank         -0.11        1.00         -0.03   -0.01      0.02   
danceability       -0.05       -0.03          1.00    0.27      0.32   
energy             -0.02       -0.01          0.27    1.00      0.65   
loudness            0.00        0.02          0.32    0.65      1.00   
acousticness       -0.06        0.02         -0.25   -0.52     -0.36   
valence            -0.03       -0.03          0.44    0.38      0.32   
tempo              -0.01        0.01         -0.16    0.10      0.07   
duration_min        0.03        0.03         -0.21   -0.14     -0.15   

              acousticness  valence  tempo  duration_min  
popularity           -0.06    -0.03  -0.01          0.03  
daily_rank            0.02    -0.03   0.01          0.03  
danceability         -0.25     0.44  -0.16         -0.21  
energy               -0.52     0.38   0.10 

In [ ]:
# ── 10. WINNING AUDIO FORMULA BY RANK TIER

In [35]:
features = ["energy", "valence", "danceability", "acousticness", "duration_min", "popularity"]
 
formula = pd.concat([
    df[df["daily_rank"] <= 10]             [features].mean().rename("Top 10"),
    df[df["daily_rank"].between(11, 50)]   [features].mean().rename("Rank 11–50"),
    df[df["daily_rank"].between(51, 100)]  [features].mean().rename("Rank 51–100"),
], axis=1).round(3)
 
print(formula)
 
 #Songs that rank highest tend to be energetic, more danceable, and very engaging for listeners.

              Top 10  Rank 11–50  Rank 51–100
energy         0.655       0.647          NaN
valence        0.560       0.543          NaN
danceability   0.684       0.674          NaN
acousticness   0.263       0.278          NaN
duration_min   3.207       3.246          NaN
popularity    78.852      75.171          NaN


In [ ]:
# ── 12. POPULARITY TREND OVER TIME

In [36]:
pop_time = df.groupby("snapshot_date")["popularity"].mean().round(2)
 
print(pop_time)
 
#Analysis emphasizes the impact of artist popularity, listener behavior, and audio characteristics on global streaming success.

snapshot_date
2023-10-18    78.72
2023-10-19    78.80
2023-10-20    78.97
2023-10-21    78.92
2023-10-22    76.52
              ...  
2025-06-07    74.37
2025-06-08    76.33
2025-06-09    76.60
2025-06-10    76.63
2025-06-11    76.64
Name: popularity, Length: 583, dtype: float64
